# Load AI-READI clinical tables

Set the dataset path below, then run the cells in order. This notebook reads your local files. It shows clinical mapping, a participant-by-feature table and CSV export. Saved outputs are cleared in this copy.

In [ ]:
from pathlib import Path
import pandas as pd
from aireadi_preprocessing import load_clinical
from aireadi_preprocessing.tables import read_table, measurement_mapping

dataset = Path("/path/to/ai-readi")
output = Path("/path/to/local-tables")

## Inspect the standardized names

The bundled [mapping](src/aireadi_preprocessing/mappings.json) matches measurement concept IDs to readable column names. Source labels may be truncated or use internal field names.

| Identifier | Source label | Standardized name |
| --- | --- | --- |
| 3004410 | import_hba1c, Hemoglobin A1c/Hemoglobin.total in | HbA1c (%) |
| 2005200344 | trails_visuospatial_executive | MoCA Trail Making Score |

`measurement_mapping` selects the bundled names present in your file and records its source units. No units or numerical values are converted.


In [ ]:
measurements = read_table(dataset / "clinical_data" / "measurement.csv")
mapping = measurement_mapping(measurements)
mapping.head()

## Load the clinical table

The package matches measurement rows by code and pivots their numeric values into columns. Participant IDs remain strings, so leading zeros are preserved. Repeated participant/code pairs use the first nonmissing value in source-file order, matching the original extraction rule.

In [ ]:
clinical = load_clinical(dataset)
clinical.head()

## Choose measurements and column names

This selection uses HbA1c, code `3004410`, with unit `%`. Change the rows to choose other measurements after checking the mapping. An absent measurement produces an empty column; a unit mismatch raises an error.

In [ ]:
selected_mapping = pd.DataFrame([
    {"concept_id": "3004410", "name": "HbA1c (%)", "unit": "%"},
])
selected = load_clinical(dataset, mapping=selected_mapping)
selected.head()

## Save and reload a CSV

Choose a local output directory outside the source dataset and repository. Keep `person_id` as text when reading the result back.

In [ ]:
output.mkdir(parents=True, exist_ok=True)
selected.to_csv(output / "clinical.csv", index=False)
selected_mapping.to_csv(output / "clinical_mapping.csv", index=False)
reloaded = pd.read_csv(output / "clinical.csv", dtype={"person_id": str})
reloaded.head()